# 02 - Demand Model

Reads the artifacts saved by `scripts/train_demand.py`, run that first:

```bash
python3 scripts/train_demand.py
```

Three models are compared on the untouched chronological test split: logistic regression
(interpretable baseline), LightGBM (diagnostic ceiling for the feature set), and the MLP
(primary, its probabilities become the simulator's world mechanics, with optional isotonic
recalibration fitted on the validation split).

Accuracy is not reported on purpose, it says nothing at a 13.7% positive rate. The metric set is
log loss, PR-AUC, Brier, ECE plus reliability diagrams, and three acceptance gates guard the RL
phase: PR-AUC at least 1.5x prevalence, price monotonicity, and an interior revenue peak.

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from airbnb_marl.analysis.plotting import PALETTE, INK_SECONDARY, apply_style, new_figure
from airbnb_marl.utils.paths import results_dir

apply_style()
ART = results_dir() / 'demand'
metrics = json.loads((ART / 'metrics.json').read_text())
arrays = np.load(ART / 'eval_arrays.npz')

MODEL_LABEL = {'lr': 'Logistic regression', 'lgbm': 'LightGBM', 'mlp': 'MLP',
               'mlp_calibrated': 'MLP + isotonic'}
MODEL_COLOR = {'lr': PALETTE[2], 'lgbm': PALETTE[1], 'mlp': PALETTE[0],
               'mlp_calibrated': PALETTE[4]}

table = pd.DataFrame(metrics['test']).T
print('final model:', metrics['final_model'])
table.round(4)

## 1. Calibration, the most important plot in this phase

The predicted probabilities are consumed as physics by the simulation: if the model says 30%,
about 30% of such nights must actually book. Points on the diagonal mean a trustworthy world model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

for name, pts in metrics['calibration_curves_test'].items():
    axes[0].plot(pts['mean_predicted'], pts['fraction_positive'], marker='o', markersize=4,
                 color=MODEL_COLOR[name], label=MODEL_LABEL[name])
lim = max(max(p['mean_predicted']) for p in metrics['calibration_curves_test'].values()) * 1.05
axes[0].plot([0, lim], [0, lim], '--', color=INK_SECONDARY, linewidth=1, label='perfect calibration')
axes[0].set_xlabel('Mean predicted probability (quantile bins)')
axes[0].set_ylabel('Empirical booking rate')
axes[0].set_title('Reliability diagram, test split')
axes[0].legend()

final = metrics['final_model']
y_test = arrays['y_test']
p_final = arrays[f'p_{final}']
bins = np.linspace(0, max(p_final.max(), 0.5), 40)
axes[1].hist(p_final[y_test == 0], bins=bins, alpha=0.65, label='not booked',
             color=PALETTE[0], edgecolor='none')
axes[1].hist(p_final[y_test == 1], bins=bins, alpha=0.65, label='booked',
             color=PALETTE[5], edgecolor='none')
axes[1].set_yscale('log')
axes[1].set_xlabel(f'Predicted probability ({MODEL_LABEL[final]})')
axes[1].set_ylabel('Nights (log)')
axes[1].set_title('Prediction distribution by outcome')
axes[1].legend()
plt.tight_layout(); plt.show()

## 2. The demand curve, elasticity correction, and gates

Mean P(booking) as the price sweeps 0.5x to 2.5x of the competitor median, all else fixed, on
20k test rows. This curve is what the RL agents will climb.

Listing prices never change within an observation window (Airbnb stopped exposing calendar prices),
so the price signal is cross sectional and partly confounded by unobserved quality: expensive flats
are also nicer flats. If the resulting revenue curve (price times probability) peaks at the price
boundary, the simulation would degenerate into agents pinned at the ceiling. In that case a
structural correction P_sim(r) = P_model(r) * r^boost is applied, calibrated to a literature
elasticity target and leaving P unchanged at r = 1, and the interior_revenue_peak gate checks the
corrected curve.

In [ ]:
dc = metrics['demand_curve_test_sample']
dc_raw = metrics['demand_curve_uncorrected']
corr = metrics['elasticity_correction']
ratios = np.array(dc['price_ratios'])
probs = np.array(dc['mean_prob'])
probs_raw = np.array(dc_raw['mean_prob'])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
axes[0].plot(ratios, probs_raw, marker='o', markersize=4, color=PALETTE[2],
             label='ML model (uncorrected)')
if corr['applied']:
    axes[0].plot(ratios, probs, marker='o', markersize=4, color=PALETTE[0],
                 label=f"simulation (boost {corr['boost']:.2f})")
axes[0].set_xlabel('Price / competitor median')
axes[0].set_ylabel('Mean P(booking)')
axes[0].set_title('Demand curve (test sample)')
axes[0].legend()

axes[1].plot(ratios, ratios * probs_raw, marker='o', markersize=4, color=PALETTE[2],
             label='uncorrected')
if corr['applied']:
    axes[1].plot(ratios, ratios * probs, marker='o', markersize=4, color=PALETTE[0],
                 label='simulation')
axes[1].axvline(dc['revenue_peak_ratio'], color=PALETTE[5], linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Price / competitor median')
axes[1].set_ylabel('Relative expected revenue (ratio x P)')
axes[1].set_title(f"Implied revenue, simulation peak at ratio {dc['revenue_peak_ratio']:.1f}")
axes[1].legend()
plt.tight_layout(); plt.show()

print('elasticity correction:', json.dumps(corr, indent=2))
print(json.dumps(metrics['gates'], indent=2))

## 3. What drives the models?

In [ ]:
coefs = pd.Series(metrics['lr_standardized_coefficients']).sort_values()
imp = pd.Series(metrics['lgbm_feature_importance']).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
colors = [PALETTE[5] if v < 0 else PALETTE[0] for v in coefs.values]
axes[0].barh(coefs.index, coefs.values, color=colors)
axes[0].set_title('LR standardized coefficients (log odds per 1 SD)')
axes[0].grid(axis='x'); axes[0].grid(axis='y', visible=False)

axes[1].barh(imp.index, imp.values, color=PALETTE[1])
axes[1].set_title('LightGBM split importance share')
axes[1].grid(axis='x'); axes[1].grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

print('price_ratio LR coefficient:', metrics['lr_standardized_coefficients']['price_ratio'],
      '(negative means higher relative price lowers booking odds)')

## Reading the results

- Gates: all three must pass before the artifact in `results/demand/` may be used as the simulator.
- MLP vs LightGBM: if the MLP sits far below LightGBM on PR-AUC, the MLP training is the problem,
  not the features. ML metrics are always for the uncorrected model since real observations do not
  obey the corrected elasticity.
- The LR price coefficient should be clearly negative. That is the elasticity sanity check quoted
  in the report; the structural correction is the simulation side answer to the residual quality
  confounding that no cross sectional model can fully remove.
- The revenue curve's interior peak previews where profit maximizing agents should end up pricing.
  The interior_revenue_peak gate guarantees the RL price bounds are not degenerate.